<a href="https://colab.research.google.com/github/chrisanuo/chrisanuo/blob/main/C_N_Ratio_split_plot_analysis_for_bulk_soil_POM_MAOM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
%load_ext rpy2.ipython

The rpy2.ipython extension is already loaded. To reload it, use:
  %reload_ext rpy2.ipython


In [ ]:
%%R

install.packages(c(
  "readxl", "dplyr", "lme4", "lmerTest",
  "emmeans", "multcomp", "multcompView",
  "openxlsx"
))

Installing packages into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)
trying URL 'https://cran.rstudio.com/src/contrib/readxl_1.5.0.tar.gz'
trying URL 'https://cran.rstudio.com/src/contrib/dplyr_1.2.1.tar.gz'
trying URL 'https://cran.rstudio.com/src/contrib/lme4_2.0-1.tar.gz'
trying URL 'https://cran.rstudio.com/src/contrib/lmerTest_3.2-1.tar.gz'
trying URL 'https://cran.rstudio.com/src/contrib/emmeans_2.0.3.tar.gz'
trying URL 'https://cran.rstudio.com/src/contrib/multcomp_1.4-30.tar.gz'
trying URL 'https://cran.rstudio.com/src/contrib/multcompView_0.1-11.tar.gz'
trying URL 'https://cran.rstudio.com/src/contrib/openxlsx_4.2.8.1.tar.gz'

The downloaded source packages are in
	‘/tmp/RtmprYop8q/downloaded_packages’


In [ ]:
%%R

library(readxl)
library(dplyr)
library(lme4)
library(lmerTest)
library(emmeans)
library(multcomp)
library(multcompView)
library(openxlsx)

# Change this to your uploaded Excel filename
file_path <- "Updated_bulk_POM_MAOM_C_N.xlsx"

sheets <- excel_sheets(file_path)

data_all <- lapply(sheets, function(sh) {
  read_excel(file_path, sheet = sh) %>%
    mutate(Depth = sh)
}) %>%
  bind_rows()

data_all <- data_all %>%
  rename(
    Tillage = Treatments,
    Rotation = `crop-rotation`,
    BulkOC = `Bulk C (g/kg)`,
    BulkN  = `Bulk N (g/kg)`,
    POMC   = `POM-C (g/kg)`,
    POMN   = `POM-N (g/kg)`,
    MAOMC  = `MAOM-C (g/kg)`,
    MAOMN  = `MAOM-N (g/kg)`,
    BulkCN = `Bulk C/N`,
    POMCN  = `POM C/N`,
    MAOMCN = `MAOM C/N`
  ) %>%
  mutate(
    Rep = factor(Rep),
    Depth = factor(
      Depth,
      levels = c("0-5", "5-15", "15-30", "30-50", "50-75", "75-100")
    ),
    Tillage = recode(Tillage, `MB plow` = "Moldboard plow"),
    Tillage = factor(
      Tillage,
      levels = c("No-till", "Chisel plow", "Moldboard plow")
    ),
    Rotation = recode(
      Rotation,
      `C-C` = "Continuous corn",
      `C-B` = "Corn-soybean",
      `B-B` = "Continuous soybean"
    ),
    Rotation = factor(
      Rotation,
      levels = c("Continuous corn", "Corn-soybean", "Continuous soybean")
    )
  )

variables <- c(
  "BulkOC", "BulkN",
  "POMC", "POMN",
  "MAOMC", "MAOMN",
  "BulkCN", "POMCN", "MAOMCN"
)

anova_results <- list()
tillage_means <- list()
rotation_means <- list()
interaction_means <- list()

for (var in variables) {

  for (dep in levels(data_all$Depth)) {

    dat <- data_all %>%
      filter(Depth == dep) %>%
      filter(!is.na(.data[[var]]))

    if (nrow(dat) == 0) next

    model <- lmer(
      as.formula(
        paste(var, "~ Rotation * Tillage + (1|Rep) + (1|Rep:Rotation)")
      ),
      data = dat
    )

    # ANOVA
    anova_tbl <- anova(model, type = 3) %>%
      as.data.frame() %>%
      mutate(
        Variable = var,
        Depth = dep,
        Effect = rownames(.)
      ) %>%
      relocate(Variable, Depth, Effect)

    anova_results[[paste(var, dep, sep = "_")]] <- anova_tbl


    # Main effect: Tillage
    emm_tillage <- emmeans(model, ~ Tillage)

    tillage_tbl <- cld(
      emm_tillage,
      adjust = "none",
      Letters = letters,
      alpha = 0.05
    ) %>%
      as.data.frame() %>%
      mutate(
        Variable = var,
        Depth = dep,
        Mean_SE = paste0(round(emmean, 3), " ± ", round(SE, 3))
      ) %>%
      relocate(
        Variable, Depth, Tillage
      )

    tillage_means[[paste(var, dep, sep = "_")]] <- tillage_tbl


    # Main effect: Rotation
    emm_rotation <- emmeans(model, ~ Rotation)

    rotation_tbl <- cld(
      emm_rotation,
      adjust = "none",
      Letters = letters,
      alpha = 0.05
    ) %>%
      as.data.frame() %>%
      mutate(
        Variable = var,
        Depth = dep,
        Mean_SE = paste0(round(emmean, 3), " ± ", round(SE, 3))
      ) %>%
      relocate(
        Variable, Depth, Rotation
      )

    rotation_means[[paste(var, dep, sep = "_")]] <- rotation_tbl


    # Interaction: Rotation × Tillage
    emm_interaction <- emmeans(model, ~ Rotation * Tillage)

    interaction_tbl <- cld(
      emm_interaction,
      adjust = "none",
      Letters = letters,
      alpha = 0.05
    ) %>%
      as.data.frame() %>%
      mutate(
        Variable = var,
        Depth = dep,
        Mean_SE = paste0(round(emmean, 3), " ± ", round(SE, 3))
      ) %>%
      relocate(
        Variable, Depth, Rotation, Tillage
      )

    interaction_means[[paste(var, dep, sep = "_")]] <- interaction_tbl
  }
}

anova_out <- bind_rows(anova_results)
tillage_out <- bind_rows(tillage_means)
rotation_out <- bind_rows(rotation_means)
interaction_out <- bind_rows(interaction_means)

wb <- createWorkbook()

addWorksheet(wb, "ANOVA")
writeData(wb, "ANOVA", anova_out)

addWorksheet(wb, "MainEffect_Tillage")
writeData(wb, "MainEffect_Tillage", tillage_out)

addWorksheet(wb, "MainEffect_Rotation")
writeData(wb, "MainEffect_Rotation", rotation_out)

addWorksheet(wb, "Interaction_TillageRotation")
writeData(wb, "Interaction_TillageRotation", interaction_out)

saveWorkbook(
  wb,
  "Full_splitplot_ANOVA_main_effects_interactions.xlsx",
  overwrite = TRUE
)

boundary (singular) fit: see help('isSingular')
Cannot use mode = "kenward-roger" because *pbkrtest* package is not installed
NOTE: Results may be misleading due to involvement in interactions
Cannot use mode = "kenward-roger" because *pbkrtest* package is not installed
NOTE: Results may be misleading due to involvement in interactions
Cannot use mode = "kenward-roger" because *pbkrtest* package is not installed
Cannot use mode = "kenward-roger" because *pbkrtest* package is not installed
NOTE: Results may be misleading due to involvement in interactions
Cannot use mode = "kenward-roger" because *pbkrtest* package is not installed
NOTE: Results may be misleading due to involvement in interactions
Cannot use mode = "kenward-roger" because *pbkrtest* package is not installed
boundary (singular) fit: see help('isSingular')
Cannot use mode = "kenward-roger" because *pbkrtest* package is not installed
NOTE: Results may be misleading due to involvement in interactions
Cannot use mode = "kenw

In [ ]:
from google.colab import files
files.download("Full_splitplot_ANOVA_main_effects_interactions.xlsx")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>